# Phase 5 — Statistical Analysis

This notebook formally tests eight pre-specified relationships for each outcome. It reports adjusted p-values and effect sizes, separates weighted description from unweighted inference, and makes no causal or predictive claims.

In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file())
sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
from IPython.display import display
from finaccess_eswatini.phase5_statistics import run

summary = run()
report_dir = project_root / 'reports' / 'phase_5'
print('Inference:', summary['design']['inference_weighting'])
print('Multiple testing:', summary['design']['multiple_testing'])
print('Significant results:', summary['result_counts'])

Inference: unweighted respondent counts
Multiple testing: Benjamini-Hochberg within each outcome
Significant results: {'account_fin': {'tests': 8, 'fdr_significant': 7}, 'account_mob': {'tests': 8, 'fdr_significant': 7}}


## Categorical associations

Pearson chi-square tests use unweighted counts. Bias-corrected Cramér's V describes association strength; weighted rate gaps provide descriptive context.

In [2]:
categorical = pd.read_csv(report_dir / 'categorical_tests.csv')
categorical_view = categorical[[
    'outcome', 'dimension_label', 'n_included', 'chi_square', 'degrees_freedom',
    'adjusted_p_value', 'effect_size', 'effect_magnitude',
    'weighted_rate_gap_pp', 'significant_fdr_0_05'
]].copy()
categorical_view[['chi_square', 'adjusted_p_value', 'effect_size', 'weighted_rate_gap_pp']] = categorical_view[[
    'chi_square', 'adjusted_p_value', 'effect_size', 'weighted_rate_gap_pp'
]].round(4)
display(categorical_view.sort_values(['outcome', 'effect_size'], ascending=[True, False]))

,outcome,dimension_label,n_included,chi_square,degrees_freedom,adjusted_p_value,effect_size,effect_magnitude,weighted_rate_gap_pp,significant_fdr_0_05
2,Financial institution account,Household income quintile,1051,80.7593,4,0.0000,0.2704,small,31.4532,True
3,Financial institution account,Workforce status,1051,74.6866,1,0.0000,0.2649,small,25.1361,True
1,Financial institution account,Education,1041,54.7618,2,0.0000,0.2252,small,45.5814,True
4,Financial institution account,Recent internet use,1051,43.0249,1,0.0000,0.2001,small,20.5384,True
6,Financial institution account,Phone type,1042,24.3918,2,0.0000,0.1467,small,18.3749,True
5,Financial institution account,Mobile phone ownership,1050,8.3469,1,0.0044,0.0837,negligible,15.8148,True
0,Financial institution account,Gender,1051,0.2978,1,0.5853,0.0000,negligible,3.4646,False
13,Mobile money account,Phone type,1042,60.2176,2,0.0000,0.2365,small,38.2246,True
9,Mobile money account,Household income quintile,1051,58.0442,4,0.0000,0.2269,small,32.3101,True
11,Mobile money account,Recent internet use,1051,54.5185,1,0.0000,0.2258,small,20.8088,True


## Numeric age comparison

The Mann–Whitney U test compares age distributions. Positive rank-biserial values indicate that respondents with the outcome tend to be older.

In [3]:
numeric = pd.read_csv(report_dir / 'numeric_tests.csv')
numeric_view = numeric[[
    'outcome', 'n_target_0', 'n_target_1', 'median_target_0', 'median_target_1',
    'adjusted_p_value', 'effect_size', 'effect_magnitude', 'direction', 'significant_fdr_0_05'
]].copy()
numeric_view[['adjusted_p_value', 'effect_size']] = numeric_view[['adjusted_p_value', 'effect_size']].round(4)
display(numeric_view)

,outcome,n_target_0,n_target_1,median_target_0,median_target_1,adjusted_p_value,effect_size,effect_magnitude,direction,significant_fdr_0_05
0,Financial institution account,514,537,30.0,36.0,0.0000,0.203,small,positive class older,True
1,Mobile money account,440,611,32.0,34.0,0.0032,0.108,small,positive class older,True


## Interpretation

Significance and practical importance are not interchangeable. The conclusions below use association language and do not determine future model features automatically.

In [4]:
for finding in summary['findings']:
    print(f'• {finding}')
print('\nAll chi-square assumption checks passed:', summary['assumptions'])

• Financial inclusion was associated after FDR adjustment with education, income quintile, workforce status, recent internet use, phone ownership, phone type, and age; gender was not associated.
• Mobile-money adoption was associated after FDR adjustment with education, income quintile, workforce status, recent internet use, phone ownership, phone type, and age; gender was not associated.
• The largest categorical effect for financial inclusion was income quintile (bias-corrected Cramér's V=0.270), closely followed by workforce status.
• The largest categorical effect for mobile money was phone type (bias-corrected Cramér's V=0.236), followed by income quintile and recent internet use.
• Included respondents were older on average for both outcomes; the age effect was small for financial inclusion (rank-biserial=0.203) and mobile money (rank-biserial=0.108).

All chi-square assumption checks passed: {'categorical_test_count': 14, 'tests_passing_expected_count_rule': 14, 'minimum_expecte